# Pipeline 3 — SQuAD Question Answering

**Owners:** Djem & Sneha  
**Branch:** `develop`

## What this notebook does
Fine-tunes both `bert-base-uncased` and `distilbert-base-uncased` on **SQuAD v1.1** —
an extractive question answering task where the model predicts the start and end position
of the answer span within a passage.

This corresponds to Table 3 in the DistilBERT paper.

Results are saved to `results/` for the final comparison notebook.

## Expected results (from paper Table 3)
| Model | EM | F1 |
|-------|-----|-----|
| BERT-base | 81.2 | 88.5 |
| DistilBERT | 77.7 | 85.8 |

> EM = Exact Match. F1 measures partial overlap between predicted and gold answer.

In [3]:
%pip install transformers datasets evaluate accelerate scikit-learn

  Using cached datasets-5.0.0-py3-none-any.whl.metadata (23 kB)
  Using cached evaluate-0.4.6-py3-none-any.whl.metadata (9.5 kB)
  Using cached accelerate-1.13.0-py3-none-any.whl.metadata (19 kB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.4/40.4 kB 1.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.7/58.7 kB 2.4 MB/s eta 0:00:00
  Using cached dill-0.4.1-py3-none-any.whl.metadata (10 kB)
  Using cached annotated_doc-0.0.4-py3-none-any.whl.metadata (6.6 kB)
  Using cached aiohappyeyeballs-2.6.2-py3-none-any.whl.metadata (5.9 kB)
  Using cached aiosignal-1.4.0-py3-none-any.whl.metadata (3.7 kB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.2/94.2 kB 2.4 MB/s eta 0:00:00
  Using cached mdurl-0.1.2-py3-none-any.whl.metadata (1.6 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.0/11.0 MB 10.2 MB/s eta 0:00:0000:0100:01
Using cached datasets-5.0.0-py3-none-any.whl (555 kB)
Using cached evaluate-0.4.6-py3-none-any.whl (84 kB)
Using cached accelerate-1.13.0-py3-no

In [1]:
# ── Install dependencies ───────────────────────────────────────────────────
!pip install -q transformers datasets evaluate accelerate scikit-learn


[notice] A new release of pip is available: 24.3.1 -> 26.1.2
[notice] To update, run: pip3 install --upgrade pip


In [3]:
%pip install torch

  Using cached mpmath-1.3.0-py3-none-any.whl.metadata (8.6 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.0/88.0 MB 7.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.4/203.4 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 16.2 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.3/6.3 MB 16.3 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.8/40.8 kB 498.2 kB/s eta 0:00:00 0:00:01
Using cached mpmath-1.3.0-py3-none-any.whl (536 kB)

[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [1]:
import sys
sys.path.append('../src')

import torch
import numpy as np
import collections
from torch.utils.data import DataLoader
from torch.optim import AdamW
from transformers import (
    AutoTokenizer,
    AutoModelForQuestionAnswering,
    get_linear_schedule_with_warmup,
)
from datasets import load_dataset
import evaluate as hf_evaluate
from tqdm import tqdm

from utils import print_model_info, save_results, count_parameters, model_size_mb

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

2026-06-09 19:56:28.974872: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-06-09 19:56:29.028453: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
/opt/micromamba/lib/python3.11/site-packages/google/protobuf/runtime_version.py:98: UserWarning: Protobuf gencode version 5.28.3 is exactly one major version older than the runtime version 6.31.1 at tensorflow/core/framework/attr_value.proto. Please update the gencode to avoid compatibility violations in the next runtime release.
  warnings.warn(
/opt/micromamba/lib/pytho

Device: cuda


In [2]:
# ── Config ─────────────────────────────────────────────────────────────────
MAX_LENGTH = 384    # standard for SQuAD
DOC_STRIDE = 128    # overlap between chunks for long passages
BATCH_SIZE = 16
EPOCHS     = 2      # paper uses 2 epochs for SQuAD
LR         = 3e-5
SEED       = 42

torch.manual_seed(SEED)

In [3]:
# ── Load SQuAD v1.1 ────────────────────────────────────────────────────────
# 87,599 train / 10,570 validation question-answer pairs
raw = load_dataset("rajpurkar/squad")
print(f"Train: {len(raw['train'])} | Validation: {len(raw['validation'])}")
print(f"\nSample question: {raw['train'][0]['question']}")
print(f"Sample answer:   {raw['train'][0]['answers']['text'][0]}")

README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/14.5M [00:00<?, ?B/s]

plain_text/validation-00000-of-00001.par(…):   0%|          | 0.00/1.82M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/87599 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/10570 [00:00<?, ? examples/s]

Train: 87599 | Validation: 10570

Sample question: To whom did the Virgin Mary allegedly appear in 1858 in Lourdes France?
Sample answer:   Saint Bernadette Soubirous


In [4]:
# ── Preprocessing ──────────────────────────────────────────────────────────
# SQuAD requires special preprocessing:
# 1. Tokenize question + context together
# 2. Find the token positions of the answer span
# 3. Handle passages longer than MAX_LENGTH using a sliding window (doc_stride)

def preprocess_train(examples, tokenizer):
    tokenized = tokenizer(
        examples['question'],
        examples['context'],
        truncation='only_second',
        max_length=MAX_LENGTH,
        stride=DOC_STRIDE,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding='max_length',
    )

    sample_map = tokenized.pop('overflow_to_sample_mapping')
    offset_map = tokenized.pop('offset_mapping')

    start_positions, end_positions = [], []

    for i, offsets in enumerate(offset_map):
        sample_idx = sample_map[i]
        answers = examples['answers'][sample_idx]
        sequence_ids = tokenized.sequence_ids(i)

        # Find where context tokens start and end
        ctx_start = sequence_ids.index(1)
        ctx_end   = len(sequence_ids) - 1 - sequence_ids[::-1].index(1)

        if len(answers['answer_start']) == 0:
            start_positions.append(0)
            end_positions.append(0)
            continue

        char_start = answers['answer_start'][0]
        char_end   = char_start + len(answers['text'][0])

        # Check if answer is within this chunk
        if offsets[ctx_start][0] > char_end or offsets[ctx_end][1] < char_start:
            start_positions.append(0)
            end_positions.append(0)
        else:
            tok_start = ctx_start
            while tok_start <= ctx_end and offsets[tok_start][0] <= char_start:
                tok_start += 1
            start_positions.append(tok_start - 1)

            tok_end = ctx_end
            while tok_end >= ctx_start and offsets[tok_end][1] >= char_end:
                tok_end -= 1
            end_positions.append(tok_end + 1)

    tokenized['start_positions'] = start_positions
    tokenized['end_positions']   = end_positions
    return tokenized


def preprocess_val(examples, tokenizer):
    tokenized = tokenizer(
        examples["question"],
        examples["context"],
        truncation="only_second",
        max_length=MAX_LENGTH,
        stride=DOC_STRIDE,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding="max_length",
    )

    sample_map = tokenized.pop("overflow_to_sample_mapping")
    tokenized["example_id"] = []

    for i in range(len(tokenized["input_ids"])):
        sample_idx = sample_map[i]
        tokenized["example_id"].append(examples["id"][sample_idx])

        sequence_ids = tokenized.sequence_ids(i)

        # Keep offsets only for context tokens.
        # Question, special tokens, and padding receive None.
        tokenized["offset_mapping"][i] = [
            offset if sequence_ids[token_idx] == 1 else None
            for token_idx, offset in enumerate(tokenized["offset_mapping"][i])
        ]

    return tokenized

In [5]:
# ── Training function (QA-specific) ───────────────────────────────────────
def train_qa_model(model, train_loader, device, epochs, lr):
    model.to(device)
    optimizer = AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    total_steps = len(train_loader) * epochs
    scheduler = get_linear_schedule_with_warmup(
        optimizer, num_warmup_steps=int(0.1 * total_steps),
        num_training_steps=total_steps
    )
    history = []

    for epoch in range(epochs):
        model.train()
        total_loss = 0
        for batch in tqdm(train_loader, desc=f'Epoch {epoch+1}/{epochs}'):
            batch = {k: v.to(device) for k, v in batch.items()
                     if k in ('input_ids', 'attention_mask', 'token_type_ids',
                               'start_positions', 'end_positions')}
            outputs = model(**batch)
            loss = outputs.loss
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            total_loss += loss.item()

        avg_loss = total_loss / len(train_loader)
        print(f'Epoch {epoch+1}: loss={avg_loss:.4f}')
        history.append({'epoch': epoch+1, 'train_loss': round(avg_loss, 4)})

    return history

In [6]:
# ── Run pipeline ───────────────────────────────────────────────────────────
def run_pipeline(model_name):
    print(f'\n{"="*50}')
    print(f'Model: {model_name} | Task: SQuAD v1.1')
    print(f'{"="*50}')

    tokenizer = AutoTokenizer.from_pretrained(model_name)

    # Tokenize train set
    train_tokenized = raw['train'].map(
        lambda x: preprocess_train(x, tokenizer),
        batched=True, remove_columns=raw['train'].column_names
    )
    cols = ['input_ids', 'attention_mask', 'start_positions', 'end_positions']
    if 'distilbert' not in model_name:
        cols.append('token_type_ids')
    train_tokenized.set_format('torch', columns=cols)
    train_loader = DataLoader(train_tokenized, batch_size=BATCH_SIZE, shuffle=True)

    model = AutoModelForQuestionAnswering.from_pretrained(model_name)
    print_model_info(model, model_name)

    history = train_qa_model(model, train_loader, DEVICE, epochs=EPOCHS, lr=LR)
    
    metrics, predictions = evaluate_qa_model(
    model=model,
    tokenizer=tokenizer,
    validation_data=raw["validation"],
    device=DEVICE,
    batch_size=BATCH_SIZE,
)
    

    total_params, _ = count_parameters(model)

    results = {
        "model": model_name,
        "task": "squad_v1",
        "exact_match": metrics["exact_match"],
        "f1": metrics["f1"],
        "total_parameters": total_params,
        "model_size_mb": model_size_mb(model),
        "inference_speed": {
            "inference_time_seconds": metrics["inference_time_seconds"],
            "samples_per_second": metrics["samples_per_second"],
            "n_validation_examples": metrics["n_validation_examples"],
            "n_tokenized_features": metrics["n_tokenized_features"],
        },
        "training_history": history,
    }

    filename = f"squad_{model_name.replace('/', '_').replace('-', '_')}.json"
    save_results(results, filename)
    return model, tokenizer, results

In [9]:
# ── Evaluation function for SQuAD ─────────────────────────────────────────
def evaluate_qa_model(
    model,
    tokenizer,
    validation_data,
    device,
    batch_size=8,
    n_best=20,
    max_answer_length=30,
):
    """
    Evaluate a question-answering model on SQuAD v1.1.

    The function:
    1. Tokenizes the validation set.
    2. Runs inference to obtain start/end logits.
    3. Converts token positions back to text answers.
    4. Computes Exact Match and F1.
    5. Measures inference time and throughput.
    """

    # Tokenize validation examples.
    val_tokenized = validation_data.map(
        lambda examples: preprocess_val(examples, tokenizer),
        batched=True,
        remove_columns=validation_data.column_names,
    )

    # The model only accepts tensor inputs.
    # example_id and offset_mapping are needed later for post-processing.
    model_inputs = val_tokenized.remove_columns(
        ["example_id", "offset_mapping"]
    )

    input_columns = ["input_ids", "attention_mask"]

    # BERT uses token_type_ids; DistilBERT does not.
    if "token_type_ids" in model_inputs.column_names:
        input_columns.append("token_type_ids")

    model_inputs.set_format(
        type="torch",
        columns=input_columns,
    )

    val_loader = DataLoader(
        model_inputs,
        batch_size=batch_size,
        shuffle=False,
    )

    # Run inference.
    all_start_logits = []
    all_end_logits = []

    model.to(device)
    model.eval()

    import time

    start_time = time.perf_counter()

    with torch.no_grad():
        for batch in tqdm(val_loader, desc="Evaluating"):
            batch = {
                key: value.to(device)
                for key, value in batch.items()
            }

            outputs = model(**batch)

            all_start_logits.append(outputs.start_logits.cpu())
            all_end_logits.append(outputs.end_logits.cpu())

    inference_time = time.perf_counter() - start_time

    all_start_logits = torch.cat(all_start_logits, dim=0).numpy()
    all_end_logits = torch.cat(all_end_logits, dim=0).numpy()

    # Map each original example ID to its position.
    example_id_to_index = {
        example_id: index
        for index, example_id in enumerate(validation_data["id"])
    }

    # One original example may produce multiple tokenized chunks.
    features_per_example = collections.defaultdict(list)

    for feature_index, feature in enumerate(val_tokenized):
        example_index = example_id_to_index[feature["example_id"]]
        features_per_example[example_index].append(feature_index)

    predictions = []

    # Convert start/end logits back to text answers.
    for example_index, example in enumerate(validation_data):
        context = example["context"]
        candidate_answers = []

        for feature_index in features_per_example[example_index]:
            start_logits = all_start_logits[feature_index]
            end_logits = all_end_logits[feature_index]
            offsets = val_tokenized[feature_index]["offset_mapping"]

            start_indexes = np.argsort(start_logits)[-n_best:][::-1]
            end_indexes = np.argsort(end_logits)[-n_best:][::-1]

            for start_index in start_indexes:
                for end_index in end_indexes:

                    # Ignore question tokens, special tokens, and padding.
                    if (
                        offsets[start_index] is None
                        or offsets[end_index] is None
                    ):
                        continue

                    # The end of the answer cannot precede the start.
                    if end_index < start_index:
                        continue

                    # Ignore answers that are too long.
                    answer_length = end_index - start_index + 1

                    if answer_length > max_answer_length:
                        continue

                    start_char = offsets[start_index][0]
                    end_char = offsets[end_index][1]

                    answer_text = context[start_char:end_char]

                    answer_score = (
                        float(start_logits[start_index])
                        + float(end_logits[end_index])
                    )

                    candidate_answers.append(
                        {
                            "text": answer_text,
                            "score": answer_score,
                        }
                    )

        if candidate_answers:
            best_answer = max(
                candidate_answers,
                key=lambda candidate: candidate["score"],
            )["text"]
        else:
            best_answer = ""

        predictions.append(
            {
                "id": example["id"],
                "prediction_text": best_answer,
            }
        )

    references = [
        {
            "id": example["id"],
            "answers": example["answers"],
        }
        for example in validation_data
    ]

    squad_metric = hf_evaluate.load("squad")

    metrics = squad_metric.compute(
        predictions=predictions,
        references=references,
    )

    metrics["inference_time_seconds"] = round(inference_time, 2)
    metrics["samples_per_second"] = round(
        len(validation_data) / inference_time,
        2,
    )
    metrics["n_validation_examples"] = len(validation_data)
    metrics["n_tokenized_features"] = len(val_tokenized)

    return metrics, predictions

In [7]:
print(torch.cuda.get_device_name(0))

Tesla V100-SXM2-32GB


In [12]:
# ── Full SQuAD runs: execute on GPU cluster only ───────────────────────────
if DEVICE.type != "cuda":
    raise RuntimeError("Full fine-tuning requires a CUDA GPU.")

bert_model, bert_tokenizer, bert_results = run_pipeline(
    "bert-base-uncased"
)

# distil_model, distil_tokenizer, distil_results = run_pipeline(
#     "distilbert-base-uncased"
# )

print(bert_results)


Model: bert-base-uncased | Task: SQuAD v1.1


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

Epoch 2/2: 100%|██████████| 5533/5533 [30:48<00:00,  2.99it/s]

Epoch 2: loss=0.7515


Map:   0%|          | 0/10570 [00:00<?, ? examples/s]

Evaluating: 100%|██████████| 674/674 [01:06<00:00, 10.18it/s]


Results saved to ../results/squad_bert_base_uncased.json
{'model': 'bert-base-uncased', 'task': 'squad_v1', 'exact_match': 81.00283822138127, 'f1': 88.39393718310535, 'total_parameters': 108893186, 'model_size_mb': 415.39, 'inference_speed': {'inference_time_seconds': 66.23, 'samples_per_second': 159.61, 'n_validation_examples': 10570, 'n_tokenized_features': 10784}, 'training_history': [{'epoch': 1, 'train_loss': 1.4759}, {'epoch': 2, 'train_loss': 0.7515}]}


In [ ]:
# ── Paper vs Ours ─────────────────────────────────────────────────────────
import pandas as pd

rows = [
    {'Model': 'BERT-base (paper)',  'EM': 80.8, 'F1': 88.5},
    {'Model': 'DistilBERT (paper)', 'EM': 77.7, 'F1': 85.8},
    {'Model': 'BERT-base (ours)',   'EM': 'TBD', 'F1': 'TBD'},
    {'Model': 'DistilBERT (ours)',  'EM': 'TBD', 'F1': 'TBD'},
]

# To compute EM/F1, run the HuggingFace squad metric on your predictions:
# metric = evaluate.load('squad')
# results = metric.compute(predictions=predictions, references=references)

print(pd.DataFrame(rows).to_string(index=False))
print('\nSee HuggingFace evaluate docs to compute EM/F1 from model predictions.')

In [11]:
del distil_model
torch.cuda.empty_cache()